# 15. Samplers and solvers — DPM-Solver++ and UniPC

DPM-Solver++의 data-prediction update와 UniPC의 UniP/UniC multistep update를 작은 model로 실행한다. UniC는 새 time point에서 이미 계산한 model output을 재사용한다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(5)
device = torch.device("cpu")


## 1. VP schedule and learned x0 predictor


In [ ]:
def alpha(t):
    return torch.cos(0.5 * math.pi * t)

def sigma(t):
    return torch.sin(0.5 * math.pi * t)

def lambda_t(t):
    return torch.log(alpha(t)) - torch.log(sigma(t))

def inverse_lambda(value):
    return 2.0 / math.pi * torch.atan(torch.exp(-value))

class DataPredictor(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
    def forward(self, x, t):
        if t.ndim == 0:
            t = t.expand(x.size(0))
        return self.net(torch.cat([x, t[:, None]], -1))

model = DataPredictor()
clean = torch.randn(16, 2)
noise = torch.randn_like(clean)
train_t = torch.linspace(0.05, 0.95, 16)
noisy = alpha(train_t)[:, None] * clean + sigma(train_t)[:, None] * noise
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
for _ in range(4):
    optimizer.zero_grad()
    loss = F.mse_loss(model(noisy, train_t), clean)
    loss.backward()
    optimizer.step()


## 2. DPM-Solver++ first- and second-order data-prediction updates


In [ ]:
def dpmpp_first_order(x_s, s, t, model_fn):
    h = lambda_t(t) - lambda_t(s)
    x0_s = model_fn(x_s, s)
    return sigma(t) / sigma(s) * x_s - alpha(t) * torch.expm1(-h) * x0_s

def dpmpp_second_order(x_s, s, t, model_fn, r1=0.5):
    lambda_s = lambda_t(s)
    h = lambda_t(t) - lambda_s
    s1 = inverse_lambda(lambda_s + r1 * h)
    x0_s = model_fn(x_s, s)
    x_s1 = sigma(s1) / sigma(s) * x_s - alpha(s1) * torch.expm1(-r1 * h) * x0_s
    x0_s1 = model_fn(x_s1, s1)
    correction = 0.5 / r1 * alpha(t) * torch.expm1(-h) * (x0_s1 - x0_s)
    return sigma(t) / sigma(s) * x_s - alpha(t) * torch.expm1(-h) * x0_s - correction

x = torch.randn(2, 2)
s = torch.tensor(0.8)
t = torch.tensor(0.6)
assert dpmpp_first_order(x, s, t, model).shape == x.shape
assert dpmpp_second_order(x, s, t, model).shape == x.shape


## 3. UniPC coefficient system


In [ ]:
def unipc_system(current_time, target_time, model_history, time_history, order, sample):
    model_s0 = model_history[-1]
    lambda_s0 = lambda_t(current_time)
    h = lambda_t(target_time) - lambda_s0
    rks = []
    d1_terms = []
    for history_offset in range(1, order):
        previous_time = time_history[-(history_offset + 1)]
        previous_model = model_history[-(history_offset + 1)]
        rk = (lambda_t(previous_time) - lambda_s0) / h
        rks.append(rk)
        d1_terms.append((previous_model - model_s0) / rk)
    rks.append(torch.ones((), device=sample.device))
    rks = torch.stack(rks)
    hh = -h
    h_phi_1 = torch.expm1(hh)
    h_phi_k = h_phi_1 / hh - 1.0
    b_h = torch.expm1(hh)
    factorial = 1.0
    rows = []
    rhs = []
    for power in range(1, order + 1):
        rows.append(rks.pow(power - 1))
        rhs.append(h_phi_k * factorial / b_h)
        factorial *= power + 1
        h_phi_k = h_phi_k / hh - 1.0 / factorial
    return {
        "model_s0": model_s0, "h_phi_1": h_phi_1, "b_h": b_h,
        "matrix": torch.stack(rows), "rhs": torch.stack(rhs),
        "d1_terms": d1_terms,
    }


## 4. UniP predictor and UniC corrector


In [ ]:
def unip_predict(sample, current_time, target_time, model_history, time_history, order):
    system = unipc_system(current_time, target_time, model_history, time_history, order, sample)
    if system["d1_terms"]:
        differences = torch.stack(system["d1_terms"])
        if order == 2:
            rho = torch.tensor([0.5], dtype=sample.dtype, device=sample.device)
        else:
            rho = torch.linalg.solve(system["matrix"][:-1, :-1], system["rhs"][:-1]).to(sample.dtype)
        residual = torch.einsum("k,kbd->bd", rho, differences)
    else:
        residual = torch.zeros_like(sample)
    base = sigma(target_time) / sigma(current_time) * sample - alpha(target_time) * system["h_phi_1"] * system["model_s0"]
    return base - alpha(target_time) * system["b_h"] * residual

def unic_correct(previous_sample, current_time, target_time, model_history, time_history, target_model_output, order):
    system = unipc_system(current_time, target_time, model_history, time_history, order, previous_sample)
    rho = torch.linalg.solve(system["matrix"], system["rhs"]).to(previous_sample.dtype)
    if system["d1_terms"]:
        previous_residual = torch.einsum("k,kbd->bd", rho[:-1], torch.stack(system["d1_terms"]))
    else:
        previous_residual = torch.zeros_like(previous_sample)
    endpoint_difference = target_model_output - system["model_s0"]
    correction = previous_residual + rho[-1] * endpoint_difference
    base = sigma(target_time) / sigma(current_time) * previous_sample - alpha(target_time) * system["h_phi_1"] * system["model_s0"]
    return base - alpha(target_time) * system["b_h"] * correction


## 5. Complete multistep execution

각 새 time point에서 model을 한 번 평가하고 그 값을 UniC와 다음 step history에 함께 사용한다.


In [ ]:
class CountedModel:
    def __init__(self, model_fn):
        self.model_fn = model_fn
        self.calls = 0
    def __call__(self, x, t):
        self.calls += 1
        return self.model_fn(x, t)

counted = CountedModel(model)
time_values = torch.linspace(0.90, 0.10, 10)
sample = torch.randn(2, 2)
model_history = []
time_history = []
current_model = counted(sample, time_values[0]).detach()
model_history.append(current_model)
time_history.append(time_values[0])
for step_index in range(len(time_values) - 1):
    current_time = time_values[step_index]
    target_time = time_values[step_index + 1]
    order = min(3, len(model_history))
    predicted = unip_predict(sample, current_time, target_time, model_history, time_history, order)
    target_model = counted(predicted, target_time).detach()
    if order >= 2:
        sample = unic_correct(sample, current_time, target_time, model_history, time_history, target_model, order)
    else:
        sample = predicted
    model_history.append(target_model)
    time_history.append(target_time)
    if len(model_history) > 3:
        model_history.pop(0)
        time_history.pop(0)
expected_model_calls = len(time_values)
assert counted.calls == expected_model_calls
assert torch.isfinite(sample).all()
print("time points:", len(time_values))
print("model calls:", counted.calls)
